In [0]:
%pip install xgboost

# Restart Python environment to load the newly installed package
dbutils.library.restartPython()

In [0]:
import mlflow
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Set Unity Catalog registry target
mlflow.set_registry_uri("databricks-uc")

# 2. Enable automatic tracking
mlflow.xgboost.autolog()

# 3. Load sample dataset and convert to Pandas DataFrames for schema signature
db_data = load_diabetes()
X = pd.DataFrame(db_data.data, columns=db_data.feature_names)
y = pd.Series(db_data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Train model inside MLflow run context
with mlflow.start_run(run_name="xgb_diabetes_poc_run") as run:
    model = xgb.XGBRegressor(
        n_estimators=100, 
        max_depth=4, 
        learning_rate=0.05, 
        random_state=42
    )
    # Fitting on DataFrames allows MLflow autolog to infer input/output signature
    model.fit(X_train, y_train)
    
    # 5. Evaluate predictions
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"Test RMSE: {rmse:.4f}")
    
    # 6. Register model to Unity Catalog (Format: catalog.schema.model_name)
    model_uri = f"runs:/{run.info.run_id}/model"
    uc_model_name = "workspace.sree_test.diabetes_xgb_model"
    
    registered_model = mlflow.register_model(
        model_uri=model_uri, 
        name=uc_model_name
    )

print(f"\nSuccessfully registered Version {registered_model.version} to {uc_model_name}")

While libraries like PyTorch or TensorFlow specialize in deep learning (neural networks), sklearn focuses on classical statistical learning for structured or tabular data (data organized in rows and columns).